In [1]:
%load_ext autoreload
%autoreload 2


from src.config import Configuration
CONFIG = Configuration()

________________________________________________________________________________________________________________________________
                                                         CONFIGURATION                                                          

Experiment description: Base experiment description
Experiment name: base_name
Model type: small
seed: 42
batch_size: 128
epochs: 100
dropout_rate: 0.5
label_smoothing: 0.1
learning_rate: 0.01
weight_decay: 0.0001
eta_min: 1e-06
momentum: 0.9
lr_reduce_factor: 0.5
lr_patience: 3
patience: 10



# Load ViT and Data

1. **Zebra** (340) - High contrast stripes.
2. **Dalmatian** (251) - Distinct spot patterns.
3. **Monarch butterfly** (323) - Clear shape and contrast against nature backgrounds.
4. **Acoustic guitar** (401) - Strong geometric lines.
5. **School bus** (779) - Large, uniform color block.
6. **Volcano** (980) - Distinct peak structure.
7. **Daisy** (985) - Radial symmetry.
8. **Peacock** (84) - Complex, detailed texture.
9. **Brain coral** (109) - High-frequency texture testing.
10. **Espresso maker** (550) - Metallic, reflective, complex object.

In [2]:
import os
from maikol_utils.file_utils import list_dir_files

all_images, _ = list_dir_files(CONFIG.attention_data, absolute_path=True)
classes = set([os.path.basename(img).split("_")[0] for img in all_images])

classes

{'bus',
 'butterfly',
 'coral',
 'daisy',
 'dalmatian',
 'espresso',
 'guitar',
 'peacock',
 'volcano',
 'zebra'}

In [3]:
from transformers import ViTModel

# Load pretrained model
model = ViTModel.from_pretrained(
    'google/vit-base-patch16-224', 
    output_attentions=True # NOTE
)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Visualization

### Get attention

In [4]:
import torch
from PIL import Image
from torchvision import transforms

to_tensor = transforms.ToTensor()

def load_image(image_path):
    image = Image.open(image_path).convert('RGB')
    image = image.resize((224, 224)) 
    
    # convert to tensor
    image = to_tensor(image).unsqueeze(0)  # (1, 3, H, W)
    return image


outputs = {
    path: model(load_image(path)) for path in all_images
}


layer_attentions = outputs[all_images[0]].attentions
print("Number of attention layers:", len(layer_attentions)) 
print("Shape of the the attention:", layer_attentions[1].shape) 
layer_attentions

Number of attention layers: 12
Shape of the the attention: torch.Size([1, 12, 197, 197])


(tensor([[[[7.8150e-01, 2.5818e-03, 2.0216e-03,  ..., 1.5438e-03,
            1.7452e-03, 2.5067e-03],
           [8.6614e-01, 8.5388e-03, 4.0925e-03,  ..., 3.1591e-04,
            3.1249e-04, 6.7172e-04],
           [8.7835e-01, 5.1903e-03, 5.3314e-03,  ..., 5.4319e-04,
            4.7520e-04, 4.4167e-04],
           ...,
           [8.3109e-01, 4.5971e-04, 5.7735e-04,  ..., 1.3255e-02,
            1.0750e-02, 8.6329e-03],
           [8.2943e-01, 4.2679e-04, 4.8859e-04,  ..., 8.5100e-03,
            1.9626e-02, 1.5173e-02],
           [8.2305e-01, 6.3369e-04, 3.1642e-04,  ..., 4.3645e-03,
            9.5894e-03, 2.4523e-02]],
 
          [[8.3236e-01, 1.2229e-03, 1.1889e-03,  ..., 1.1748e-03,
            1.1767e-03, 1.1165e-03],
           [9.5435e-03, 9.4359e-03, 1.7237e-02,  ..., 6.0732e-03,
            7.2430e-03, 3.2197e-03],
           [9.9039e-03, 1.5641e-02, 1.0468e-02,  ..., 3.4491e-03,
            3.8904e-03, 7.0678e-03],
           ...,
           [4.9754e-02, 9.0909e-03, 5.

### Layer CLS token attentio over all the paches at layer $l$

- We have 224×224 image separed into 14×14 paches: total of 196 paches.
- We had 197 bc it was $196 \text{(PATCHES)} + 1 \text{(CLS)}$

In [5]:
from typing import Literal
def get_attention_at_layer(attention, layer_idx, type: Literal["mean", "max", "heads"] = "mean"):
    # Extracts the specific layer, averages across all heads, 
    # and isolates the CLS token's attention to the 196 spatial patches
    # return attention[layer_idx][0][0, 1:, 1:]
    if type == "mean":
        return attention[layer_idx][0].mean(dim=0)[0, 1:]
    elif type == "heads":
        return attention[layer_idx][0][:, 0, 1:]
    elif type == "max":
        # .max(dim=0)[0] extracts the max values (PyTorch returns values and indices), then we slice CLS to spatial patches
        return attention[layer_idx][0].max(dim=0)[0][0, 1:]


get_attention_at_layer(layer_attentions, 0).shape

torch.Size([196])

In [6]:
import os
import importlib
import cv2
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from PIL import Image


def _to_attention_grid(attn_1d):
    grid_size = int(np.sqrt(attn_1d.shape[-1]))
    return attn_1d.reshape(grid_size, grid_size)


def _resize_attention_map(attn_1d, target_size=(224, 224)):
    attn_grid = _to_attention_grid(attn_1d)
    return cv2.resize(attn_grid, target_size, interpolation=cv2.INTER_CUBIC)


def _normalize_map(attn_map):
    attn_min = float(attn_map.min())
    attn_max = float(attn_map.max())
    if attn_max - attn_min < 1e-8:
        return np.zeros_like(attn_map, dtype=np.float32)
    return ((attn_map - attn_min) / (attn_max - attn_min)).astype(np.float32)


def _extract_label_from_filename(image_path):
    # Approximate predicted label from filename prefix (e.g., "zebra_*.jpg" -> "zebra")
    return os.path.basename(image_path).split("_")[0]


def _build_single_map_figure(output, image_path, mode="mean", layer_idx=0):
    attention = output.attentions
    num_layers = len(attention)

    if layer_idx < 0 or layer_idx >= num_layers:
        raise ValueError(f"layer_idx must be in [0, {num_layers - 1}]")

    img = np.array(Image.open(image_path).convert("RGB").resize((224, 224)))

    attn_1d = get_attention_at_layer(attention, layer_idx, type=mode).detach().cpu().numpy()
    attn_map = _resize_attention_map(attn_1d, target_size=(224, 224))

    fig = go.Figure()
    fig.add_trace(go.Image(z=img))
    fig.add_trace(go.Heatmap(
        z=attn_map,
        opacity=0.5,
        colorscale="Jet",
        showscale=False
    ))

    predicted_label = _extract_label_from_filename(image_path)

    fig.update_layout(
        title=f"ViT Attention ({mode.upper()}) - Layer {layer_idx}",
        margin=dict(l=40, r=40, t=40, b=40),
        annotations=[
            dict(
                x=0.01,
                y=0.99,
                xref="paper",
                yref="paper",
                text=f"Predicted: {predicted_label}",
                showarrow=False,
                xanchor="left",
                yanchor="top",
                font=dict(size=12, color="white"),
                bgcolor="rgba(0,0,0,0.6)",
                bordercolor="rgba(255,255,255,0.6)",
                borderwidth=1,
                borderpad=3
            )
        ]
    )

    return fig


def _build_heads_subplot_figure(output, image_path, layer_idx=0):
    attention = output.attentions
    num_layers = len(attention)

    if layer_idx < 0 or layer_idx >= num_layers:
        raise ValueError(f"layer_idx must be in [0, {num_layers - 1}]")

    img = np.array(Image.open(image_path).convert("RGB").resize((224, 224)))
    heads_attn = get_attention_at_layer(attention, layer_idx, type="heads").detach().cpu().numpy()  # (num_heads, 196)

    num_heads = heads_attn.shape[0]
    rows = 2
    cols = int(np.ceil(num_heads / rows))

    fig = make_subplots(
        rows=rows,
        cols=cols,
        subplot_titles=[f"Head {h}" for h in range(num_heads)],
        horizontal_spacing=0.02,
        vertical_spacing=0.10
    )

    for h in range(num_heads):
        row = h // cols + 1
        col = h % cols + 1

        attn_map = _resize_attention_map(heads_attn[h], target_size=(224, 224))
        norm_map = _normalize_map(attn_map)

        heat_u8 = np.uint8(norm_map * 255.0)
        heat_rgb = cv2.cvtColor(cv2.applyColorMap(heat_u8, cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)

        # Blend attention with the original image for each head panel.
        overlay = cv2.addWeighted(img, 0.55, heat_rgb, 0.45, 0)

        fig.add_trace(go.Image(z=overlay), row=row, col=col)

    predicted_label = _extract_label_from_filename(image_path)

    fig.update_layout(
        title=f"ViT Attention Heads (Layer {layer_idx})",
        margin=dict(l=20, r=20, t=60, b=20),
        showlegend=False,
        annotations=[
            dict(
                x=0.01,
                y=1.06,
                xref="paper",
                yref="paper",
                text=f"Predicted: {predicted_label}",
                showarrow=False,
                xanchor="left",
                yanchor="top",
                font=dict(size=12, color="white"),
                bgcolor="rgba(0,0,0,0.6)",
                bordercolor="rgba(255,255,255,0.6)",
                borderwidth=1,
                borderpad=3
            )
        ]
    )

    fig.update_xaxes(showticklabels=False)
    fig.update_yaxes(showticklabels=False)

    return fig


def visualize_attention(output, image_path, mode="mean", layer_idx=0):
    attention = output.attentions
    num_layers = len(attention)

    widgets_spec = importlib.util.find_spec("ipywidgets")

    # Interactive controls when ipywidgets is available.
    if widgets_spec is not None:
        widgets = importlib.import_module("ipywidgets")
        ipy_display = importlib.import_module("IPython.display")

        mode_buttons = widgets.ToggleButtons(
            options=[("Mean", "mean"), ("Max", "max"), ("Heads", "heads")],
            value=mode if mode in {"mean", "max", "heads"} else "mean",
            description="Mode:"
        )

        layer_slider = widgets.IntSlider(
            value=max(0, min(layer_idx, num_layers - 1)),
            min=0,
            max=num_layers - 1,
            step=1,
            description="Layer:",
            continuous_update=False
        )

        out = widgets.Output()

        def _render(*_):
            with out:
                ipy_display.clear_output(wait=True)
                if mode_buttons.value in {"mean", "max"}:
                    fig = _build_single_map_figure(
                        output=output,
                        image_path=image_path,
                        mode=mode_buttons.value,
                        layer_idx=layer_slider.value
                    )
                else:
                    fig = _build_heads_subplot_figure(
                        output=output,
                        image_path=image_path,
                        layer_idx=layer_slider.value
                    )
                fig.show()

        mode_buttons.observe(_render, names="value")
        layer_slider.observe(_render, names="value")

        controls = widgets.HBox([mode_buttons, layer_slider])
        ipy_display.display(widgets.VBox([controls, out]))
        _render()
        return

    # Fallback without widgets: render one selected view.
    if mode in {"mean", "max"}:
        fig = _build_single_map_figure(output=output, image_path=image_path, mode=mode, layer_idx=layer_idx)
    else:
        fig = _build_heads_subplot_figure(output=output, image_path=image_path, layer_idx=layer_idx)

    fig.show()
    print("ipywidgets not installed: pass mode='mean'|'max'|'heads' and layer_idx manually.")

In [ ]:
i = 16
visualize_attention(outputs[all_images[i]], all_images[i])

No. Plotting the raw weight matrix directly will not overlay the attention on the image. The raw matrix represents patch-to-patch attention (e.g., shape `197x197`), which is unreadable as a spatial image. 

To visualize a single layer's attention on the original image, you must isolate the **CLS token's attention** and reshape it.

Here is the practical pipeline to achieve this:

1. **Extract** the attention weights for a specific layer.
2. **Average** the weights across all attention heads (if you aren't analyzing individual heads).
3. **Slice** the attention values from the CLS token (index 0) to all other spatial patches (indices 1 to end).
4. **Reshape** that 1D array into a 2D spatial grid (e.g., $14 \times 14$).
5. **Resize** that grid back to your original image resolution (e.g., $224 \times 224$) and overlay it.

Here is the minimal code implementation for that logic:

* **Attention rollout:**
    * Feed image into ViT
    * Attention layers $[A^l_h]$: layer $l$ has $h$ attention heads with size $(N+1)\times(N+1)$ ($N$ is the number of image patch tokens + 1 CLS token)
    * Aggregate (can be done with mean, max, min...) the attention over heads so that $[A^l_h]$ is reduced to $[A^l]$
    * Compute the attention rollout (estimating the flow of attention in the ViT network):
$$A_{rollout}^l = (A^l + I)A_{rollout}^{l-1}, \text{ for } i > 1$$
$$A_{rollout}^1 = A^1 + I$$